# Cleaning validation

This notebook reads the existing raw and processed files. It does not alter the cleaning rules or write data.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

raw_paths = {
    'March': ROOT / 'data/raw/listings-2026-03-16.csv.gz',
    'June': ROOT / 'data/raw/listings-2026-06-15.csv.gz',
}
raw = {name: pd.read_csv(path, low_memory=False) for name, path in raw_paths.items()}
cleaned = pd.read_csv(ROOT / 'data/processed/la_listings_cleaned.csv', low_memory=False)
matched = pd.read_csv(ROOT / 'data/processed/la_matched_listing_price_trends.csv', low_memory=False)

def missing_or_blank(series):
    return series.isna() | series.astype('string').str.strip().eq('')

clean = {
    'March': cleaned.loc[cleaned['snapshot_name'].eq('march_2026')],
    'June': cleaned.loc[cleaned['snapshot_name'].eq('june_2026')],
}


## Row counts and missing values

In [ ]:
rows = []
for snapshot in ['March', 'June']:
    source, output = raw[snapshot], clean[snapshot]
    rows.append({
        'snapshot': snapshot,
        'raw_rows': len(source), 'cleaned_rows': len(output),
        'raw_price_missing': missing_or_blank(source['price']).sum(),
        'clean_price_missing': output['price_usd'].isna().sum(),
        'raw_bedrooms_missing': source['bedrooms'].isna().sum(),
        'clean_bedrooms_missing': output['bedrooms'].isna().sum(),
        'raw_neighbourhood_missing': missing_or_blank(source['neighbourhood_cleansed']).sum(),
        'clean_neighbourhood_missing': output['neighbourhood'].isna().sum(),
        'raw_room_type_missing': missing_or_blank(source['room_type']).sum(),
        'clean_room_type_missing': output['room_type'].isna().sum(),
    })
pd.DataFrame(rows)

## Duplicate and valid-price checks

In [ ]:
duplicates = []
price_summary = []
for snapshot in ['March', 'June']:
    source, output = raw[snapshot], clean[snapshot]
    duplicates.append({
        'snapshot': snapshot,
        'exact_duplicate_rows': source.duplicated().sum(),
        'rows_with_duplicate_listing_id': source['id'].duplicated(keep=False).sum(),
    })
    price = output.loc[output['has_valid_price'], 'price_usd']
    p25, p75 = price.quantile([0.25, 0.75])
    price_summary.append({
        'snapshot': snapshot, 'valid_price_count': len(price),
        'min_usd': price.min(), 'median_usd': price.median(),
        'iqr_usd': p75 - p25, 'max_usd': price.max(),
    })

display(pd.DataFrame(duplicates))
display(pd.DataFrame(price_summary).round(2))

## Matched-listing checks

In [ ]:
pd.DataFrame([{
    'matched_listing_ids': len(matched),
    'valid_price_both': matched['has_valid_price_both'].sum(),
    'valid_price_both_and_segment_unchanged': (
        matched['has_valid_price_both'] & ~matched['segment_changed']
    ).sum(),
}])

## Five validated record checks

In [ ]:
checks = pd.DataFrame([
    ('march_2026', 1452068118494812958, 'price missing', 'price_usd null; has_valid_price false'),
    ('june_2026', 9140, 'price missing', 'price_usd null; has_valid_price false'),
    ('march_2026', 49731, 'bedrooms missing; price $114.14', 'bedrooms null; bedroom_segment Unknown'),
    ('june_2026', 2708, 'bedrooms missing; price $67.22', 'bedrooms null; bedroom_segment Unknown'),
    ('march_2026', 1452093968341982976, 'bedrooms 0; price $315.00', 'bedrooms 0; bedroom_segment Studio'),
], columns=['snapshot_name', 'listing_id', 'original', 'expected'])

actual = cleaned.merge(checks[['snapshot_name', 'listing_id']], on=['snapshot_name', 'listing_id'])
actual['actual'] = (
    'price_usd=' + actual['price_usd'].astype('string')
    + '; has_valid_price=' + actual['has_valid_price'].astype(str)
    + '; bedrooms=' + actual['bedrooms'].astype('string')
    + '; bedroom_segment=' + actual['bedroom_segment']
)
checks.merge(actual[['snapshot_name', 'listing_id', 'actual']], on=['snapshot_name', 'listing_id'])